# Edinburgh Airbnb Dataset Familiarization

## Dataset Context

This notebook examines the Inside Airbnb dataset for Edinburgh, Scotland,
using the snapshot published on 23 June 2026.

## Objectives

- Review the available source files
- Document dataset sizes and schemas
- Inspect representative sample records
- Identify candidate primary and foreign keys
- Validate relationships between datasets
- Investigate structural differences between detailed and summary files
- Record assumptions and dataset limitations

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display


pd.set_option("display.max_columns", 150)
pd.set_option("display.max_colwidth", 120)

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

metadata_dir = project_root / "data" / "metadata"

dataset_inventory = pd.read_csv(
    metadata_dir / "dataset_inventory.csv"
)

schema_report = pd.read_csv(
    metadata_dir / "schema_report.csv"
)

print(f"Project root: {project_root}")

display(dataset_inventory)

Project root: d:\Airbnb-Market-Intelligence\airbnb-market-intelligence-edinburgh


,dataset,file_name,file_format,row_count,column_count,size_bytes,size_mb
0,listings_detailed,listings_detailed.csv,CSV,6244,90,17274112,16.47
1,calendar_detailed,calendar_detailed.csv,CSV,2284170,5,78798221,75.15
2,reviews_detailed,reviews_detailed.csv,CSV,676263,6,207665110,198.04
3,listings_summary,listings.csv,CSV,6258,19,1199265,1.14
4,reviews_summary,reviews.csv,CSV,676263,2,15117568,14.42
5,neighbourhoods,neighbourhoods.csv,CSV,111,2,2568,0.00
6,neighbourhoods_geojson,neighbourhoods.geojson,GeoJSON,111,3,325835,0.31


## 1. Dataset Schemas

The following tables document the inferred column names and data types for
each available Edinburgh dataset.

In [2]:
schema_columns = [
    column
    for column in ["column_name", "column_type", "null", "key"]
    if column in schema_report.columns
]

for dataset_name in schema_report["dataset"].unique():
    print(f"\n{'=' * 80}")
    print(f"SCHEMA: {dataset_name}")
    print(f"{'=' * 80}")

    dataset_schema = (
        schema_report.loc[
            schema_report["dataset"] == dataset_name,
            schema_columns,
        ]
        .reset_index(drop=True)
    )

    display(dataset_schema)


SCHEMA: listings_detailed


,column_name,column_type,null,key
0,id,BIGINT,YES,NaN
1,listing_url,VARCHAR,YES,NaN
2,scrape_id,BIGINT,YES,NaN
3,last_scraped,DATE,YES,NaN
4,source,VARCHAR,YES,NaN
...,...,...,...,...
85,calculated_host_listings_count,BIGINT,YES,NaN
86,calculated_host_listings_count_entire_homes,BIGINT,YES,NaN
87,calculated_host_listings_count_private_rooms,BIGINT,YES,NaN
88,calculated_host_listings_count_shared_rooms,BIGINT,YES,NaN



SCHEMA: calendar_detailed


,column_name,column_type,null,key
0,listing_id,BIGINT,YES,NaN
1,date,DATE,YES,NaN
2,available,BOOLEAN,YES,NaN
3,minimum_nights,BIGINT,YES,NaN
4,maximum_nights,BIGINT,YES,NaN



SCHEMA: reviews_detailed


...

,column_name,column_type,null,key
0,listing_id,BIGINT,YES,NaN
1,date,DATE,YES,NaN



SCHEMA: neighbourhoods


,column_name,column_type,null,key
0,neighbourhood_group,VARCHAR,YES,NaN
1,neighbourhood,VARCHAR,YES,NaN



SCHEMA: neighbourhoods_geojson


,column_name,column_type,null,key
0,neighbourhood,str,UNKNOWN,NaN
1,neighbourhood_group,NaN,UNKNOWN,NaN
2,geometry,MultiPolygon,UNKNOWN,NaN


## 2. Representative Sample Records

Only small samples are loaded during familiarization. Large datasets such as
the calendar and detailed reviews files are queried with DuckDB rather than
loaded completely into notebook memory.

In [3]:
import json

import duckdb


raw_dir = (
    project_root
    / "data"
    / "raw"
    / "edinburgh"
    / "2026-06-23"
)

interim_dir = (
    project_root
    / "data"
    / "interim"
    / "edinburgh"
    / "2026-06-23"
)

csv_files = {
    "listings_detailed": interim_dir / "listings_detailed.csv",
    "calendar_detailed": interim_dir / "calendar_detailed.csv",
    "reviews_detailed": interim_dir / "reviews_detailed.csv",
    "listings_summary": raw_dir / "listings.csv",
    "reviews_summary": raw_dir / "reviews.csv",
    "neighbourhoods": raw_dir / "neighbourhoods.csv",
}

geojson_file = raw_dir / "neighbourhoods.geojson"

connection = duckdb.connect()


def csv_source(file_path: Path) -> str:
    """Create a DuckDB CSV expression for a source file."""

    safe_path = file_path.as_posix().replace("'", "''")

    return (
        "read_csv_auto("
        f"'{safe_path}', "
        "header=true, "
        "sample_size=200000"
        ")"
    )


for dataset_name, file_path in csv_files.items():
    assert file_path.exists(), f"Missing file: {file_path}"

assert geojson_file.exists(), f"Missing file: {geojson_file}"

print("All source files are available.")

All source files are available.


In [4]:
for dataset_name, file_path in csv_files.items():
    print(f"\n{'=' * 80}")
    print(f"SAMPLE RECORDS: {dataset_name}")
    print(f"{'=' * 80}")

    sample = connection.execute(
        f"""
        SELECT *
        FROM {csv_source(file_path)}
        LIMIT 3
        """
    ).fetchdf()

    display(sample)


SAMPLE RECORDS: listings_detailed


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_profile_id,host_profile_url,host_name,host_since,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,price_quote_checkin_date,price_quote_checkout_date,price_quote_total_price,price_quote_price_per_night,price_quote_raw,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,15420,https://www.airbnb.com/rooms/15420,20260623040725,2026-07-02,previous scrape,Georgian Boutique Apt City Centre,"Stunning, spacious ground floor apartment minutes from Princes Street. Your own ‘home from home’ in an historic pro...",None,https://a0.muscache.com/pictures/cf69631f-4194-4020-ae63-9d745a90ec74.jpg,60423,https://www.airbnb.com/users/show/60423,1462507973102154618,https://www.airbnb.com/users/profile/1462507973102154618,Charlotte,None,16,6,15,6,"Edinburgh, United Kingdom","I have a background in property, having worked as an interior designer and on a number of renovations. Born in Wale...",None,None,None,True,None,https://a0.muscache.com/im/users/60423/profile_pic/1326194735/original.jpg?aki_policy=profile_x_medium,None,1,None,None,True,True,None,"Old Town, Princes Street and Leith Street",None,55.957590,-3.188050,Entire rental unit,Entire home/apt,2,NaN,1 bath,1,<NA>,"[""Sound system"", ""Bed linens"", ""Single level home"", ""Fire extinguisher"", ""Free dryer \u2013 In unit"", ""Iron"", ""Stain...",$225.50,2026-10-26,2026-10-28,451.0,225.50,"{""quote"": {""taxes"": null, ""currency"": ""GBP"", ""date_match"": null, ""service_fee"": null, ""total_price"": ""451.00"", ""clea...",2,30,2,4,30,30,3.0,30.0,None,True,0,1,4,28,2026-07-02,704,73,6,26,67,255,57503,2011-01-18,2026-06-28,4.98,4.99,4.97,4.98,4.99,4.98,4.92,EH-68481-F,None,1,1,0,0,3.74
1,24288,https://www.airbnb.com/rooms/24288,20260623040725,2026-06-23,city scrape,"Cool central Loft, sleeps 4, 2 double bed+en-suite",Upper level of duplex. Boho rustic-chic former warehouse Loft located in lively University quarter of City centre fo...,None,https://a0.muscache.com/pictures/3460007/887314c6_original.jpg,46498,https://www.airbnb.com/users/show/46498,1462507566386729212,https://www.airbnb.com/users/profile/1462507566386729212,Gordon,None,16,8,15,5,"Edinburgh, United Kingdom",Principal\nStudio DuB\nArchitecture & Planning industry\n2006 – Present,None,None,None,True,None,https://a0.muscache.com/im/users/46498/profile_pic/1259125328/original.jpg?aki_policy=profile_x_medium,None,1,None,None,True,True,None,"Canongate, Southside and Dumbiedykes",None,55.944983,-3.185293,Entire loft,Entire home/apt,4,1.5,1.5 baths,2,2,"[""Paid parking off premises"", ""Wifi"", ""Iron"", ""Dedicated workspace"", ""Wine glasses"", ""Toaster"", ""B


SAMPLE RECORDS: calendar_detailed


,listing_id,date,available,minimum_nights,maximum_nights
0,15420,2026-07-02,False,3,30
1,15420,2026-07-03,False,3,30
2,15420,2026-07-04,False,3,30



SAMPLE RECORDS: reviews_detailed


...

,id,name,host_id,host_profile_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license
0,15420,Georgian Boutique Apt City Centre,60423,1462507973102154618,Charlotte,None,"Old Town, Princes Street and Leith Street",55.957590,-3.188050,Entire home/apt,226,2,704,2026-06-28,3.74,1,28,73,EH-68481-F
1,24288,"Cool central Loft, sleeps 4, 2 double bed+en-suite",46498,1462507566386729212,Gordon,None,"Canongate, Southside and Dumbiedykes",55.944983,-3.185293,Entire home/apt,202,3,433,2026-06-07,2.26,1,176,46,NaN
2,38628,Edinburgh Holiday Let,165635,1462511471634092578,Trish,None,Joppa,55.942150,-3.096400,Entire home/apt,129,1,76,2026-05-25,0.52,1,258,1,EH-70886-F



SAMPLE RECORDS: reviews_summary


,listing_id,date
0,15420,2011-01-18
1,15420,2011-01-31
2,15420,2011-04-19



SAMPLE RECORDS: neighbourhoods


,neighbourhood_group,neighbourhood
0,None,Abbeyhill
1,None,Baberton and Juniper Green
2,None,Balerno and Bonnington Village


In [5]:
with geojson_file.open("r", encoding="utf-8") as file:
    neighbourhood_geojson = json.load(file)

features = neighbourhood_geojson.get("features", [])

geojson_summary = {
    "geojson_type": neighbourhood_geojson.get("type"),
    "feature_count": len(features),
    "first_feature_property_names": (
        list(features[0].get("properties", {}).keys())
        if features
        else []
    ),
    "geometry_types": sorted(
        {
            feature.get("geometry", {}).get("type", "Unknown")
            for feature in features
        }
    ),
}

display(pd.DataFrame([geojson_summary]))

if features:
    print("First feature properties:")
    display(pd.DataFrame([features[0].get("properties", {})]))

,geojson_type,feature_count,first_feature_property_names,geometry_types
0,FeatureCollection,111,"[neighbourhood, neighbourhood_group]",[MultiPolygon]


First feature properties:


,neighbourhood,neighbourhood_group
0,Balerno and Bonnington Village,None


## 3. Detailed and Summary Listings Comparison

The detailed listings file contains 6,244 rows, while the summary listings
file contains 6,258 rows. Listing identifiers are compared to determine
whether the difference is caused by records appearing in only one source.

In [6]:
detailed_listings_source = csv_source(
    csv_files["listings_detailed"]
)

summary_listings_source = csv_source(
    csv_files["listings_summary"]
)

listing_id_differences = connection.execute(
    f"""
    WITH detailed AS (
        SELECT DISTINCT CAST(id AS VARCHAR) AS listing_id
        FROM {detailed_listings_source}
    ),
    summary AS (
        SELECT DISTINCT CAST(id AS VARCHAR) AS listing_id
        FROM {summary_listings_source}
    )
    SELECT
        COALESCE(detailed.listing_id, summary.listing_id) AS listing_id,
        CASE
            WHEN detailed.listing_id IS NULL THEN 'summary_only'
            WHEN summary.listing_id IS NULL THEN 'detailed_only'
        END AS difference_type
    FROM detailed
    FULL OUTER JOIN summary
        ON detailed.listing_id = summary.listing_id
    WHERE detailed.listing_id IS NULL
       OR summary.listing_id IS NULL
    ORDER BY difference_type, listing_id
    """
).fetchdf()

print("Difference counts:")

display(
    listing_id_differences["difference_type"]
    .value_counts()
    .rename_axis("difference_type")
    .reset_index(name="listing_count")
)

print("Listing identifiers with differences:")

display(listing_id_differences)

Difference counts:


,difference_type,listing_count
0,summary_only,14


Listing identifiers with differences:


,listing_id,difference_type
0,1350610720950598524,summary_only
1,1350611066733418146,summary_only
2,1521613509401953853,summary_only
3,1541201389377646246,summary_only
4,1613027651593659341,summary_only
5,1615124206326713859,summary_only
6,1615124206411544751,summary_only
7,1615124206718245558,summary_only
8,1663668945350139051,summary_only
9,1663677026022166859,summary_only


### Listing Coverage Finding

The identifier comparison found 14 listings in the summary listings file that
are not present in the detailed listings file. No listings were found only in
the detailed file.

This confirms that the two files do not have identical listing coverage,
despite belonging to the same Edinburgh snapshot. The difference may result
from different generation rules, filtering, or publication timing.

The detailed listings file will be used as the primary source for rich listing
attributes because it contains 90 columns. The 14 summary-only listings will
remain documented as a data-quality finding rather than being silently removed.

## 4. Initial Inventory Observations

1. The Edinburgh snapshot contains seven available source datasets.

2. The detailed listings file contains 6,244 rows and 90 columns, while the
   summary listings file contains 6,258 rows and 19 columns.

3. The 14-row difference between the detailed and summary listing files
   requires identifier-level investigation. Records will not be removed until
   the reason for the difference is understood.

4. The detailed and summary reviews files both contain 676,263 rows. The
   detailed file contains additional review metadata and text, while the
   summary file provides a smaller structure for time-based analysis.

5. The neighbourhood CSV and GeoJSON files both contain 111 records, providing
   an initial indication of matching geographic coverage.

6. The calendar dataset is the largest by row count, containing 2,284,170
   daily listing records.

7. Detailed review data is the largest source by storage size, at
   approximately 198 MB.

8. The detailed listings dataset contains the broadest schema, with 90
   attributes describing listings, hosts, prices, locations, amenities and
   review performance.

## Candidate Primary and Composite Key Validation

Candidate keys are checked for null values and duplicates before being used in
joins or dimensional modelling.

Expected candidate keys:

| Dataset | Candidate key |
|---|---|
| Detailed listings | `id` |
| Summary listings | `id` |
| Calendar | `listing_id`, `date` |
| Detailed reviews | `id` |
| Summary reviews | `listing_id`, `date` — to be tested |

In [7]:
def check_single_column_key(source: str, key_column: str):
    """Check nulls and uniqueness for a candidate single-column key."""

    return connection.execute(
        f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT({key_column}) AS non_null_key_rows,
            COUNT(*) - COUNT({key_column}) AS null_key_rows,
            COUNT(DISTINCT {key_column}) AS distinct_key_values,
            COUNT(*) - COUNT(DISTINCT {key_column}) AS duplicate_difference
        FROM {source}
        """
    ).fetchdf()


detailed_listing_key_check = check_single_column_key(
    detailed_listings_source,
    "id",
)

summary_listing_key_check = check_single_column_key(
    summary_listings_source,
    "id",
)

print("Detailed listings key check:")
display(detailed_listing_key_check)

print("Summary listings key check:")
display(summary_listing_key_check)

Detailed listings key check:


,total_rows,non_null_key_rows,null_key_rows,distinct_key_values,duplicate_difference
0,6244,6244,0,6244,0


Summary listings key check:


,total_rows,non_null_key_rows,null_key_rows,distinct_key_values,duplicate_difference
0,6258,6258,0,6258,0


In [8]:
calendar_source = csv_source(
    csv_files["calendar_detailed"]
)

calendar_key_summary = connection.execute(
    f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(
            CASE
                WHEN listing_id IS NULL OR date IS NULL THEN 1
                ELSE 0
            END
        ) AS null_composite_key_rows
    FROM {calendar_source}
    """
).fetchdf()

calendar_duplicate_groups = connection.execute(
    f"""
    SELECT COUNT(*) AS duplicate_key_groups
    FROM (
        SELECT
            listing_id,
            date,
            COUNT(*) AS row_count
        FROM {calendar_source}
        GROUP BY listing_id, date
        HAVING COUNT(*) > 1
    )
    """
).fetchdf()

print("Calendar composite-key null check:")
display(calendar_key_summary)

print("Calendar duplicate listing-date groups:")
display(calendar_duplicate_groups)

Calendar composite-key null check:


,total_rows,null_composite_key_rows
0,2284170,0.0


Calendar duplicate listing-date groups:


,duplicate_key_groups
0,0


In [9]:
detailed_reviews_source = csv_source(
    csv_files["reviews_detailed"]
)

detailed_review_key_check = check_single_column_key(
    detailed_reviews_source,
    "id",
)

print("Detailed reviews key check:")
display(detailed_review_key_check)

Detailed reviews key check:


,total_rows,non_null_key_rows,null_key_rows,distinct_key_values,duplicate_difference
0,676263,676263,0,676263,0


In [10]:
summary_reviews_source = csv_source(
    csv_files["reviews_summary"]
)

summary_review_key_summary = connection.execute(
    f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(
            CASE
                WHEN listing_id IS NULL OR date IS NULL THEN 1
                ELSE 0
            END
        ) AS null_candidate_key_rows
    FROM {summary_reviews_source}
    """
).fetchdf()

summary_review_duplicate_groups = connection.execute(
    f"""
    SELECT COUNT(*) AS duplicate_listing_date_groups
    FROM (
        SELECT
            listing_id,
            date,
            COUNT(*) AS row_count
        FROM {summary_reviews_source}
        GROUP BY listing_id, date
        HAVING COUNT(*) > 1
    )
    """
).fetchdf()

display(summary_review_key_summary)
display(summary_review_duplicate_groups)

,total_rows,null_candidate_key_rows
0,676263,0.0


,duplicate_listing_date_groups
0,2486


## Foreign-Key Relationship Validation

The following relationships are tested:

- `calendar.listing_id → listings.id`
- `reviews_detailed.listing_id → listings.id`
- `reviews_summary.listing_id → listings.id`

Coverage is checked against both detailed and summary listings because the two
listing files have different record coverage.

In [11]:
relationship_report = connection.execute(
    f"""
    WITH detailed_listing_ids AS (
        SELECT DISTINCT CAST(id AS VARCHAR) AS listing_id
        FROM {detailed_listings_source}
        WHERE id IS NOT NULL
    ),
    summary_listing_ids AS (
        SELECT DISTINCT CAST(id AS VARCHAR) AS listing_id
        FROM {summary_listings_source}
        WHERE id IS NOT NULL
    ),
    calendar_ids AS (
        SELECT CAST(listing_id AS VARCHAR) AS listing_id
        FROM {calendar_source}
    ),
    detailed_review_ids AS (
        SELECT CAST(listing_id AS VARCHAR) AS listing_id
        FROM {detailed_reviews_source}
    ),
    summary_review_ids AS (
        SELECT CAST(listing_id AS VARCHAR) AS listing_id
        FROM {summary_reviews_source}
    )

    SELECT
        'calendar_to_detailed_listings' AS relationship,
        COUNT(*) AS child_rows,
        SUM(
            CASE WHEN parent.listing_id IS NULL THEN 1 ELSE 0 END
        ) AS unmatched_rows,
        ROUND(
            100.0 * SUM(
                CASE WHEN parent.listing_id IS NULL THEN 1 ELSE 0 END
            ) / COUNT(*),
            4
        ) AS unmatched_percentage
    FROM calendar_ids child
    LEFT JOIN detailed_listing_ids parent
        ON child.listing_id = parent.listing_id

    UNION ALL

    SELECT
        'calendar_to_summary_listings',
        COUNT(*),
        SUM(
            CASE WHEN parent.listing_id IS NULL THEN 1 ELSE 0 END
        ),
        ROUND(
            100.0 * SUM(
                CASE WHEN parent.listing_id IS NULL THEN 1 ELSE 0 END
            ) / COUNT(*),
            4
        )
    FROM calendar_ids child
    LEFT JOIN summary_listing_ids parent
        ON child.listing_id = parent.listing_id

    UNION ALL

    SELECT
        'detailed_reviews_to_detailed_listings',
        COUNT(*),
        SUM(
            CASE WHEN parent.listing_id IS NULL THEN 1 ELSE 0 END
        ),
        ROUND(
            100.0 * SUM(
                CASE WHEN parent.listing_id IS NULL THEN 1 ELSE 0 END
            ) / COUNT(*),
            4
        )
    FROM detailed_review_ids child
    LEFT JOIN detailed_listing_ids parent
        ON child.listing_id = parent.listing_id

    UNION ALL

    SELECT
        'detailed_reviews_to_summary_listings',
        COUNT(*),
        SUM(
            CASE WHEN parent.listing_id IS NULL THEN 1 ELSE 0 END
        ),
        ROUND(
            100.0 * SUM(
                CASE WHEN parent.listing_id IS NULL THEN 1 ELSE 0 END
            ) / COUNT(*),
            4
        )
    FROM detailed_review_ids child
    LEFT JOIN summary_listing_ids parent
        ON child.listing_id = parent.listing_id

    UNION ALL

    SELECT
        'summary_reviews_to_detailed_listings',
        COUNT(*),
        SUM(
            CASE WHEN parent.listing_id IS NULL THEN 1 ELSE 0 END
        ),
        ROUND(
            100.0 * SUM(
                CASE WHEN parent.listing_id IS NULL THEN 1 ELSE 0 END
            ) / COUNT(*),
            4
        )
    FROM summary_review_ids child
    LEFT JOIN detailed_listing_ids parent
        ON child.listing_id = parent.listing_id

    UNION ALL

    SELECT
        'summary_reviews_to_summary_listings',
        COUNT(*),
        SUM(
            CASE WHEN parent.listing_id IS NULL THEN 1 ELSE 0 END
        ),
        ROUND(
            100.0 * SUM(
                CASE WHEN parent.listing_id IS NULL THEN 1 ELSE 0 END
            ) / COUNT(*),
            4
        )
    FROM summary_review_ids child
    LEFT JOIN summary_listing_ids parent
        ON child.listing_id = parent.listing_id
    """
).fetchdf()

display(relationship_report)

,relationship,child_rows,unmatched_rows,unmatched_percentage
0,calendar_to_detailed_listings,2284170,5110.0,0.2237
1,calendar_to_summary_listings,2284170,0.0,0.0000
2,detailed_reviews_to_detailed_listings,676263,642.0,0.0949
3,detailed_reviews_to_summary_listings,676263,0.0,0.0000
4,summary_reviews_to_detailed_listings,676263,642.0,0.0949
5,summary_reviews_to_summary_listings,676263,0.0,0.0000
